# VoltWay Israel — Data Cleaning & Exploration

Final Project of the course -  Data Analyst. Data quality research and preprocessing

**Important:** datafiles located in the folder `data/`, containing source CSV. Output data will be placed in Database, and `clean_data/` folder


## 1. Loading and validating every csv

In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("data")

dfs = {
    file.stem: pd.read_csv(file)
    for file in sorted(DATA_DIR.glob("*.csv"))
}

for name, df in dfs.items():
    print(f"{name}: {df.shape}")

columns_vehicle_registry_en: (22, 2)
ev_by_district: (90, 4)
lookup_districts: (27, 4)
lookup_fuel: (6, 2)
lookup_makers: (225, 5)
lookup_ownership: (5, 2)
lookup_wltp_body_type: (14, 2)
lookup_wltp_technology: (4, 2)
vehicle_registry_2024_2026: (861136, 22)
wltp_models_2024_2026: (19847, 94)


## 2. Checking data types, missing data and duplicates

In [2]:
for name, df in dfs.items():
    print(f"\n{'=' * 60}")
    print(f"TABLE: {name}")
    print(f"{'=' * 60}")

    print("\nColumns and types:")
    print(df.dtypes)

    print("\nMissing values:")
    print(df.isna().sum()[lambda x: x > 0])

    print("\nDuplicate rows:")
    print(df.duplicated().sum())

hebrew_columns = []




TABLE: columns_vehicle_registry_en

Columns and types:
column        str
meaning_en    str
dtype: object

Missing values:
Series([], dtype: int64)

Duplicate rows:
0

TABLE: ev_by_district

Columns and types:
baalut        str
mahoz_nm      str
nafa_nm       str
car_num     int64
dtype: object

Missing values:
mahoz_nm    2
nafa_nm     8
dtype: int64

Duplicate rows:
0

TABLE: lookup_districts

Columns and types:
mahoz_nm          str
nafa_nm           str
district_en       str
subdistrict_en    str
dtype: object

Missing values:
nafa_nm           5
subdistrict_en    5
dtype: int64

Duplicate rows:
0

TABLE: lookup_fuel

Columns and types:
sug_delek_nm    str
fuel_en         str
dtype: object

Missing values:
Series([], dtype: int64)

Duplicate rows:
0

TABLE: lookup_makers

Columns and types:
tozeret_cd               int64
tozeret_nm                 str
maker_en                   str
production_country_en      str
translation_note           str
dtype: object

Missing values:
producti

## 3. Registry: ID, production years, fuel types

In [3]:
vehicles = dfs["vehicle_registry_2024_2026"].copy()

print("\nTotal rows:", len(vehicles))
print("Unique IDs:", vehicles["mispar_rechev"].nunique())
print("Duplicate IDs:", vehicles["mispar_rechev"].duplicated().sum())

print("\nProduction years:")
print(vehicles["shnat_yitzur"].value_counts().sort_index())

print("\nFuel types:")
print(vehicles["sug_delek_nm"].value_counts(dropna=False))
hebrew_columns=[]
for col in vehicles.select_dtypes(include="str").columns:
    if vehicles[col].astype("string").str.contains(
        r"[\u0590-\u05FF]", na=False
    ).any():
        hebrew_columns.append(col)

print("Columns with Hebrew text:")
for col in hebrew_columns:
    print(col, "—", vehicles[col].nunique(), "unique values")


Total rows: 861136
Unique IDs: 861136
Duplicate IDs: 0

Production years:
shnat_yitzur
2024    272806
2025    299876
2026    288454
Name: count, dtype: int64

Fuel types:
sug_delek_nm
בנזין         538588
חשמל          163393
חשמל/בנזין    111528
דיזל           47432
חשמל/דיזל        165
גפמ"              30
Name: count, dtype: int64
Columns with Hebrew text:
tozeret_nm — 225 unique values
degem_manoa — 1241 unique values
baalut — 5 unique values
tzeva_rechev — 94 unique values
sug_delek_nm — 6 unique values


## 4. Missing pollution categories by fuel type

In [4]:
print(
    pd.crosstab(
        vehicles["sug_delek_nm"],
        vehicles["kvutzat_zihum"].isna()
    )
)

kvutzat_zihum   False   True 
sug_delek_nm                 
בנזין          538588       0
גפמ"               30       0
דיזל            47432       0
חשמל                0  163393
חשמל/בנזין     111528       0
חשמל/דיזל         165       0


## 5. Missing first registration months

In [5]:
missing_dates = vehicles[
    vehicles["moed_aliya_lakvish"].isna()
]

print(
    pd.crosstab(
        missing_dates["shnat_yitzur"],
        missing_dates["sug_delek_nm"],
        margins=True
    )
)

sug_delek_nm  בנזין  דיזל  All
shnat_yitzur                  
2024             28    26   54
2025            175    56  231
2026            106    42  148
All             309   124  433


## 6. Registration months format check

In [6]:
dates = vehicles["moed_aliya_lakvish"].dropna()

invalid_dates = dates[
    ~dates.str.fullmatch(r"\d{4}-(?:[1-9]|1[0-2])")
]

print("Invalid date formats:", len(invalid_dates))
print(invalid_dates.value_counts().head(20))

Invalid date formats: 0
Series([], Name: count, dtype: int64)


## 7. Registration date and quarter

In [7]:
vehicles["first_registration_date"] = pd.to_datetime(
    vehicles["moed_aliya_lakvish"],
    format="%Y-%m",
    errors="coerce"
)

vehicles["registration_quarter"] = (
    vehicles["first_registration_date"].dt.to_period("Q")
)

print(vehicles[[
    "moed_aliya_lakvish",
    "first_registration_date",
    "registration_quarter"
]].head(10))

print("Missing dates:",
      vehicles["first_registration_date"].isna().sum())

  moed_aliya_lakvish first_registration_date registration_quarter
0            2025-10              2025-10-01               2025Q4
1             2026-9              2026-09-01               2026Q3
2             2026-2              2026-02-01               2026Q1
3             2026-1              2026-01-01               2026Q1
4             2026-1              2026-01-01               2026Q1
5             2026-3              2026-03-01               2026Q1
6             2026-6              2026-06-01               2026Q2
7            2025-11              2025-11-01               2025Q4
8             2026-1              2026-01-01               2026Q1
9             2024-6              2024-06-01               2024Q2
Missing dates: 433


## 8. Investigating missing trim level data

In [8]:
print(
    vehicles.loc[
        vehicles["ramat_gimur"].isna(),
        "tozeret_nm"
    ].value_counts().head(20)
)

tozeret_nm
דונגפנג סין       2002
איון סין          1149
פוטון סין          932
גי.איי.סי סין      813
וויה סין           700
אאודי גרמניה       529
ליפמוטור סין       177
ג'יי איי סי        157
באייק סין          136
מקסוס סין           97
אאודי ספרד          70
קוואן סין           50
פורשה גרמניה        38
פרארי               32
ג'יוואן סין         13
רנו צרפת             9
אס דבליו אמ          7
אסטון מרטין אנ       5
סנטרו סין            5
מודרן סין            2
Name: count, dtype: int64


## 9. Missimg safety data

In [9]:
print(
    pd.crosstab(
        vehicles["sug_delek_nm"],
        vehicles["ramat_eivzur_betihuty"].isna(),
        margins=True
    )
)

ramat_eivzur_betihuty   False   True     All
sug_delek_nm                                
בנזין                  519643  18945  538588
גפמ"                       29      1      30
דיזל                    40034   7398   47432
חשמל                   161229   2164  163393
חשמל/בנזין             111363    165  111528
חשמל/דיזל                 165      0     165
All                    832463  28673  861136


## 10. Other tech fields mising data

In [10]:
cols = ["zmig_ahori", "horaat_rishum"]

for col in cols:
    print(f"\n{col}")
    print(
        vehicles.loc[
            vehicles[col].isna(),
            ["tozeret_nm", "sug_delek_nm", "shnat_yitzur"]
        ].value_counts().head(10)
    )

print("\nVehicle colors:")
print("Unique color codes:", vehicles["tzeva_cd"].nunique())
print("Unique color names:", vehicles["tzeva_rechev"].nunique())

print(
    vehicles[["tzeva_cd", "tzeva_rechev"]]
    .drop_duplicates()
    .sort_values("tzeva_cd")
    .to_string(index=False)
)

color_translations = {
    1: "Pearl Green",
    2: "Pearl Black",
    3: "Pearl Blue",
    4: "Metallic Yellow",
    5: "Metallic Melange",
    6: "Dark Metallic Gray",
    7: "Metallic Silver Blue",
    8: "Metallic Copper",
    9: "Metallic Olive Green",
    10: "Black",
    11: "Metallic Black",
    13: "Metallic Olive",
    14: "Metallic Reddish",
    15: "Light Metallic Gray",
    16: "Bright Yellow",
    17: "Bright Red",
    19: "Matte Tangerine",
    20: "Gray",
    21: "Steel Gray",
    22: "Dark Gray",
    23: "Light Gray",
    24: "Bronze Gray",
    25: "Silver",
    26: "Platinum",
    27: "Metallic Gray",
    28: "Metallic Greenish",
    29: "Metallic Silver",
    30: "Blue",
    31: "Dark Blue",
    32: "Light Blue",
    33: "Sky Blue",
    34: "Turquoise",
    36: "Metallic Blue",
    37: "Metallic Blue Gray",
    38: "Metallic Charcoal Blue",
    39: "Metallic Sky Blue",
    40: "Green",
    41: "Light Green",
    42: "Dark Green",
    43: "Greenish",
    44: "Light Greenish",
    45: "Olive Green",
    46: "Metallic Green",
    47: "Metallic Burgundy",
    48: "Silver Green",
    49: "Metallic Gold Green",
    50: "Red",
    51: "Light Red",
    52: "Dark Wine Red",
    53: "Purple",
    54: "Burgundy",
    55: "Pink",
    56: "Metallic Red",
    57: "Eggplant",
    58: "Copper",
    59: "Metallic Rose",
    60: "Yellow",
    61: "Lemon Yellow",
    62: "Sahara",
    63: "Mustard",
    64: "Orange",
    65: "Gold",
    66: "Beige",
    67: "Dark Beige",
    68: "Cream",
    69: "Metallic Gold",
    70: "Brown",
    71: "Light Brown",
    72: "Dark Brown",
    73: "Golden",
    74: "Aqua Green",
    75: "Dark Purple",
    76: "Greenish Silver",
    77: "Classic Red",
    78: "Metallic Turquoise",
    79: "Sea Silver",
    80: "Ivory White",
    81: "Ivory",
    82: "Other",
    83: "Multicolor",
    84: "Light Metallic Turquoise",
    85: "Light Purple",
    86: "Millennium Silver",
    87: "Metallic Beige",
    88: "Metallic Coffee",
    91: "Crystal Blue",
    92: "Bronze",
    93: "Dark Silver",
    94: "Light Silver",
    95: "Turquoise Green",
    96: "Sea Green",
    97: "Metallic Bluish Silver",
    98: "Red Black",
    99: "Dark Metallic Silver"
}

vehicles["color_en"] = vehicles["tzeva_cd"].map(color_translations)

print("Missing translations:", vehicles["color_en"].isna().sum())
print("Unique translated colors:", vehicles["color_en"].nunique())


zmig_ahori
tozeret_nm      sug_delek_nm  shnat_yitzur
מרצדס בנץ ארהב  חשמל          2024            1
טויוטה צרפת     בנזין         2024            1
                              2025            1
סיטרואן צרפת    דיזל          2025            1
Name: count, dtype: int64

horaat_rishum
tozeret_nm      sug_delek_nm  shnat_yitzur
יונדאי ד. קורי  בנזין         2024            13
טויוטה יפן      בנזין         2024             9
סקודה צ'כיה     בנזין         2024             7
מרצדס בנץ גרמנ  דיזל          2024             6
קיה ד. קוריאה   בנזין         2024             5
רנו ד. קוריאה   בנזין         2024             4
טויוטה צרפת     בנזין         2024             4
סקודה צ'כיה     חשמל/בנזין    2024             4
טויוטה טורקיה   בנזין         2024             4
טויוטה ארהב"    בנזין         2024             3
Name: count, dtype: int64

Vehicle colors:
Unique color codes: 94
Unique color names: 94
 tzeva_cd      tzeva_rechev
        1        ירוק פנינה
        2        שחור פנינה
      

## 11. Fuel types 

In [11]:
print(dfs["lookup_fuel"].to_string(index=False))

sug_delek_nm                  fuel_en
       בנזין                   Petrol
        חשמל                 Electric
  חשמל/בנזין Hybrid (electric/petrol)
        דיזל                   Diesel
   חשמל/דיזל Hybrid (electric/diesel)
        גפמ"                      LPG


## 12. English fuel types merge

In [12]:
fuel = dfs["lookup_fuel"]

vehicles = vehicles.merge(
    fuel,
    on="sug_delek_nm",
    how="left",
    validate="many_to_one"
)

print("Rows after merge:", len(vehicles))
print("Missing translations:", vehicles["fuel_en"].isna().sum())
print(vehicles["fuel_en"].value_counts())

ownership = dfs["lookup_ownership"]

vehicles = vehicles.merge(
    ownership,
    on="baalut",
    how="left",
    validate="many_to_one"
)

print("Rows:", len(vehicles))
print("Missing translations:", vehicles["ownership_en"].isna().sum())

print(vehicles["ownership_en"].value_counts())

Rows after merge: 861136
Missing translations: 0
fuel_en
Petrol                      538588
Electric                    163393
Hybrid (electric/petrol)    111528
Diesel                       47432
Hybrid (electric/diesel)       165
LPG                             30
Name: count, dtype: int64
Rows: 861136
Missing translations: 0
ownership_en
Private    488357
Leasing    246318
Company     85053
Rental      29978
Dealer      11430
Name: count, dtype: int64


## 13. Car makers 

In [13]:
makers = dfs["lookup_makers"].copy()

print(makers.head(10).to_string(index=False))

print("\nDuplicate maker codes:")
print(makers["tozeret_cd"].duplicated().sum())

print("\nDuplicate code + name:")
print(
    makers.duplicated(
        subset=["tozeret_cd", "tozeret_nm"]
    ).sum()
)

 tozeret_cd     tozeret_nm   maker_en production_country_en translation_note
         19   אאודי גרמניה       Audi               Germany              NaN
         21  אאודי סלובקיה       Audi              Slovakia              NaN
         32  ב מ וו אנגליה        BMW        United Kingdom              NaN
         33    אופל גרמניה       Opel               Germany              NaN
         57     די אס צרפת         DS                France              NaN
         63    רנו סלובניה    Renault              Slovenia              NaN
         74 אלפא רומיאו_אי Alfa Romeo                 Italy              NaN
         76    אופל אנגליה       Opel        United Kingdom              NaN
         82 פולקסווגן פורט Volkswagen              Portugal              NaN
        101  איווקו איטליה      Iveco                 Italy              NaN

Duplicate maker codes:
0

Duplicate code + name:
0


## 14. Merge car makers and countries

In [14]:
vehicles = vehicles.merge(
    makers[["tozeret_cd", "maker_en", "production_country_en"]],
    on="tozeret_cd",
    how="left",
    validate="many_to_one"
)

print("Rows:", len(vehicles))
print("Missing makers:", vehicles["maker_en"].isna().sum())
print("Missing countries:", vehicles["production_country_en"].isna().sum())

print("\nTop 10 makers:")
print(vehicles["maker_en"].value_counts().head(10))

Rows: 861136
Missing makers: 0
Missing countries: 86723

Top 10 makers:
maker_en
Toyota        109084
Hyundai        84530
Kia            71937
Chery          67671
Skoda          57893
BYD            49019
Jaecoo         47735
MG             27797
Mazda          24395
Mitsubishi     19855
Name: count, dtype: int64


## 15. Car makers with no country info

In [15]:
missing_countries = vehicles[
    vehicles["production_country_en"].isna()
]

print(
    missing_countries.groupby(
        ["tozeret_cd", "maker_en", "tozeret_nm"]
    ).size().sort_values(ascending=False)
)

tozeret_cd  maker_en           tozeret_nm    
481         Hyundai            יונדאי ד. קורי    71571
1346        Lynk & Co          לינק אנד קו        7005
1566        KGM (KG Mobility)  קיי ג'י מובילי     5019
1381        FAW                אף איי דבליו       1544
1526        Jeep (Avenger)     ג'יפ אוונג'ר       1320
1213        JIC                ג'יי איי סי         157
1461        WMI                דבליו אם איי         42
770         Ferrari            פרארי                35
1569        SWM                אס דבליו אמ          30
dtype: int64


## 16. Untranslated countries info

In [16]:
print(
    makers.loc[
        makers["production_country_en"].isna(),
        ["tozeret_cd", "tozeret_nm", "maker_en", "translation_note"]
    ].to_string(index=False)
)

 tozeret_cd     tozeret_nm          maker_en                      translation_note
        481 יונדאי ד. קורי           Hyundai country part 'ד. קורי' not translated
        770          פרארי           Ferrari                                   NaN
       1213    ג'יי איי סי               JIC  transliteration - brand not verified
       1346    לינק אנד קו         Lynk & Co                                   NaN
       1381   אף איי דבליו               FAW                                   NaN
       1461   דבליו אם איי               WMI  transliteration - brand not verified
       1526   ג'יפ אוונג'ר    Jeep (Avenger)                                   NaN
       1566 קיי ג'י מובילי KGM (KG Mobility)                                   NaN
       1569    אס דבליו אמ               SWM                                   NaN


## 17. MIssing countries check with WLTP

In [17]:
wltp = dfs["wltp_models_2024_2026"]

missing_codes = makers.loc[
    makers["production_country_en"].isna(),
    "tozeret_cd"
]

print(
    wltp.loc[
        wltp["tozeret_cd"].isin(missing_codes),
        ["tozeret_cd", "tozeret_nm", "tozeret_eretz_nm"]
    ].drop_duplicates().to_string(index=False)
)

 tozeret_cd     tozeret_nm tozeret_eretz_nm
        481 יונדאי ד. קורי        ד. קוריאה
       1381   אף איי דבליו              סין
       1566 קיי ג'י מובילי        ד. קוריאה
       1569    אס דבליו אמ              סין
        770          פרארי           איטליה
       1213    ג'יי איי סי              סין
       1346    לינק אנד קו              סין
       1461   דבליו אם איי              סין
       1526   ג'יפ אוונג'ר            פולין


## 18. Check of WLTP data integrity

In [18]:
print(
    wltp.groupby("tozeret_cd")["tozeret_eretz_nm"]
    .nunique()
    .loc[missing_codes]
)

tozeret_cd
481     1
770     1
1213    1
1346    1
1381    1
1461    1
1526    1
1566    1
1569    1
Name: tozeret_eretz_nm, dtype: int64


## 19.  WLTP countries dictionary

In [19]:
country_map = (
    wltp[["tozeret_cd", "tozeret_eretz_nm"]]
    .drop_duplicates()
    .set_index("tozeret_cd")["tozeret_eretz_nm"]
)

print(country_map.loc[missing_codes])

tozeret_cd
481     ד. קוריאה
770        איטליה
1213          סין
1346          סין
1381          סין
1461          סין
1526        פולין
1566    ד. קוריאה
1569          סין
Name: tozeret_eretz_nm, dtype: str


## 20. Restoring Missing Production Countries

In [20]:

country_translations = {
    "ד. קוריאה": "South Korea",
    "סין": "China",
    "איטליה": "Italy",
    "פולין": "Poland"
}

missing_codes = makers.loc[
    makers["production_country_en"].isna(),
    "tozeret_cd"
]

country_lookup = (
    wltp[wltp["tozeret_cd"].isin(missing_codes)]
    [["tozeret_cd", "tozeret_eretz_nm"]]
    .drop_duplicates()
)

assert country_lookup["tozeret_cd"].is_unique
assert set(country_lookup["tozeret_cd"]) == set(missing_codes)

country_lookup["country_en"] = (
    country_lookup["tozeret_eretz_nm"]
    .map(country_translations)
)

assert country_lookup["country_en"].notna().all()

country_by_code = country_lookup.set_index(
    "tozeret_cd"
)["country_en"]

before = vehicles["production_country_en"].isna().sum()

vehicles["production_country_en"] = (
    vehicles["production_country_en"].fillna(
        vehicles["tozeret_cd"].map(country_by_code)
    )
)

makers["production_country_en"] = (
    makers["production_country_en"].fillna(
        makers["tozeret_cd"].map(country_by_code)
    )
)

after = vehicles["production_country_en"].isna().sum()

print("Missing before:", before)
print("Missing after:", after)
print("Restored:", before - after)


Missing before: 86723
Missing after: 0
Restored: 86723


## 21. Cleaning results validation

In [21]:
assert len(vehicles) == 861136
assert vehicles["mispar_rechev"].is_unique

assert vehicles["fuel_en"].notna().all()
assert vehicles["maker_en"].notna().all()
assert vehicles["production_country_en"].notna().all()

assert vehicles["first_registration_date"].isna().sum() == 433

print("All validation checks passed!")
print("Missing countries:", vehicles["production_country_en"].isna().sum())

All validation checks passed!
Missing countries: 0


## 22. Check districts data and WLTP integrity

In [22]:

districts = dfs["ev_by_district"].copy()

print("Total rows:", len(districts))

print("\nMissing values:")
print(districts.isna().sum())

print("\nRows with missing district or subdistrict:")
print(
    districts[
        districts["mahoz_nm"].isna() |
        districts["nafa_nm"].isna()
    ].to_string(index=False)
)

print(
    districts.groupby(
        "mahoz_nm",
        dropna=False
    )["car_num"].sum().sort_values(ascending=False)
)

print(
    dfs["lookup_districts"].to_string(index=False)
)


Total rows: 90

Missing values:
baalut      0
mahoz_nm    2
nafa_nm     8
car_num     0
dtype: int64

Rows with missing district or subdistrict:
baalut mahoz_nm nafa_nm  car_num
  פרטי      NaN     NaN      486
  חברה    הצפון     NaN      118
  חברה    המרכז     NaN       24
  חברה      NaN     NaN        9
  חברה  תל אביב     NaN        8
  חברה     חיפה     NaN        7
  חברה  ירושלים     NaN        3
  סוחר    הצפון     NaN        1
mahoz_nm
המרכז                 102525
תל אביב                42878
הצפון                  34889
הדרום                  32263
חיפה                   28819
ירושלים                15873
אזור יהודה ושומרון     11442
NaN                      495
Name: car_num, dtype: int64
          mahoz_nm      nafa_nm            district_en   subdistrict_en
           תל אביב      תל אביב               Tel Aviv         Tel Aviv
             המרכז    פתח תקווה                Central      Petah Tikva
             המרכז       רחובות                Central          Rehovot
 

## 23. District data check

In [23]:
lookup_districts = dfs["lookup_districts"]

print(
    lookup_districts.duplicated(
        subset=["mahoz_nm", "nafa_nm"]
    ).sum()
)

0


## 24. Merging english districts names

In [24]:
districts = districts.merge(
    lookup_districts,
    on=["mahoz_nm", "nafa_nm"],
    how="left",
    validate="many_to_one"
)

print("Rows:", len(districts))
print("Missing English districts:", districts["district_en"].isna().sum())
print("Missing English subdistricts:", districts["subdistrict_en"].isna().sum())

Rows: 90
Missing English districts: 2
Missing English subdistricts: 8


## 25. WLTP validation

In [25]:
print("Rows:", len(wltp))

print("\nModel codes:")
print(wltp["degem_cd"].nunique())

print("\nDrive technologies:")
print(wltp["technologiat_hanaa_nm"].value_counts(dropna=False))

print(dfs["lookup_wltp_technology"].to_string(index=False))

body_types = dfs["lookup_wltp_body_type"]

print(body_types.to_string(index=False))

print("\nDuplicate body types:")
print(body_types["merkav"].duplicated().sum())

Rows: 19847

Model codes:
4358

Drive technologies:
technologiat_hanaa_nm
הנעה רגילה      8128
היברידי רגיל    5121
רכב חשמלי       4299
PLUG IN         2299
Name: count, dtype: int64
technologiat_hanaa_nm    drive_technology_en
           הנעה רגילה    Conventional engine
         היברידי רגיל Hybrid (self-charging)
            רכב חשמלי         Electric (BEV)
              PLUG IN         Plug-in hybrid
    merkav     body_type_en
  פנאי-שטח  SUV / crossover
      סדאן            Sedan
     הצ'בק        Hatchback
      קופה            Coupe
   קבריולט      Convertible
       MPV              MPV
     קומבי        Combi van
  משא אחוד Pickup / utility
   תא כפול       Double cab
    סטיישן    Station wagon
סגור/משלוח        Panel van
ואן/נוסעים    Passenger van
   תא בודד       Single cab
       שדה         Off-road

Duplicate body types:
0


## 26. English names for technology and body types

In [26]:
technology = dfs["lookup_wltp_technology"]

wltp = wltp.merge(
    technology,
    on="technologiat_hanaa_nm",
    how="left",
    validate="many_to_one"
)

print("Rows:", len(wltp))
print("Missing translations:", wltp["drive_technology_en"].isna().sum())

print(wltp["drive_technology_en"].value_counts())

wltp = wltp.merge(
    body_types,
    on="merkav",
    how="left",
    validate="many_to_one"
)

print("Rows:", len(wltp))
print("Missing body types:", wltp["body_type_en"].isna().sum())

print(wltp["body_type_en"].value_counts(dropna=False))

Rows: 19847
Missing translations: 0
drive_technology_en
Conventional engine       8128
Hybrid (self-charging)    5121
Electric (BEV)            4299
Plug-in hybrid            2299
Name: count, dtype: int64
Rows: 19847
Missing body types: 1
body_type_en
SUV / crossover     11222
Sedan                3250
Hatchback            1453
Coupe                1370
Convertible           734
MPV                   646
Combi van             310
Pickup / utility      225
Double cab            222
Station wagon         178
Panel van             134
Passenger van          77
Single cab             15
Off-road               10
NaN                     1
Name: count, dtype: int64


## 27. WLTP keys validation

In [27]:
print("Duplicate model codes:")
print(wltp["degem_cd"].duplicated().sum())

print("\nDuplicate maker + model:")
print(
    wltp.duplicated(
        subset=["tozeret_cd", "degem_cd"]
    ).sum()
)

print("\nUnique maker + model combinations:")
print(
    wltp[["tozeret_cd", "degem_cd"]]
    .drop_duplicates()
    .shape[0]
)

duplicates = wltp[
    wltp.duplicated(
        subset=["tozeret_cd", "degem_cd"],
        keep=False
    )
]

print(
    duplicates[
        [
            "tozeret_cd",
            "degem_cd",
            "degem_nm",
            "drive_technology_en",
            "body_type_en"
        ]
    ].head(20).to_string(index=False)
)

Duplicate model codes:
15489

Duplicate maker + model:
5235

Unique maker + model combinations:
14612
 tozeret_cd  degem_cd       degem_nm drive_technology_en    body_type_en
         19      1383         8YARWY Conventional engine       Combi van
         21       179         4MN0N2 Conventional engine SUV / crossover
         74       256       949AXA2A Conventional engine SUV / crossover
        143       359           21CF      Electric (BEV) SUV / crossover
        143       359           21CF      Electric (BEV) SUV / crossover
        152       491        243.608      Electric (BEV) SUV / crossover
        187        69 GUN125L-DTTSXW Conventional engine      Double cab
        187        69 GUN125L-DTTSXW Conventional engine      Double cab
        187        71 GUN135L-DTTSXW Conventional engine      Double cab
        187        71 GUN135L-DTTSXW Conventional engine      Double cab
        187        72 GUN125L-DTTSXW Conventional engine      Double cab
        187        72 

## 28. Cleaning Log

We checked all 10 CSV files.

1. We found no duplicate rows in the files. The vehicle IDs are also unique.

2. We found 163,393 missing values in kvutzat_zihum. All of them are electric cars. We decided to keep these values as they are.

3. We found 433 missing registration dates. These cars use petrol or diesel. We kept them, but we will not use them in the quarterly analysis.

4. We checked the date format. All existing dates are correct. We converted them to dates and added quarters.

5. We found 6,932 missing trim levels. We kept them because we do not need this field for our analysis.

6. We found 28,673 missing values in safety data. We kept them.

7. We found 4 missing values in zmig_ahori and 93 in horaat_rishum. We kept them.

8. We added English fuel names. All vehicles have a fuel translation.

9. We added English manufacturer names. All manufacturers were matched.

10. We found 86,723 cars without a production country. We checked the WLTP file and found the missing countries. We filled these values.

11. We found 2 missing districts and 8 missing subdistricts. We kept these values because we don't know the exact locations. No rows were deleted.

12. We found repeated maker and model codes in WLTP. Full rows are unique. We kept all records.


13. We translated the vehicle colors from Hebrew to English. We kept the original color names and codes.

14. We added English ownership names. All vehicles have a translation.


We started with 861,136 vehicles.
After cleaning, we still have 861,136 vehicles.
No vehicle rows were deleted.

## 29. Clean data export

In [28]:

OUTPUT_DIR = Path("clean_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

export_tables = dfs.copy()

export_tables["vehicle_registry_2024_2026"] = vehicles
export_tables["lookup_makers"] = makers
export_tables["wltp_models_2024_2026"] = wltp
export_tables["ev_by_district"] = districts

for name, df in export_tables.items():
    df.to_csv(
        OUTPUT_DIR / f"{name}.csv",
        index=False,
        encoding="utf-8-sig" # Hebrew characters support
    )

    print(f"Saved: {name} ({len(df)} rows)")


Saved: columns_vehicle_registry_en (22 rows)
Saved: ev_by_district (90 rows)
Saved: lookup_districts (27 rows)
Saved: lookup_fuel (6 rows)
Saved: lookup_makers (225 rows)
Saved: lookup_ownership (5 rows)
Saved: lookup_wltp_body_type (14 rows)
Saved: lookup_wltp_technology (4 rows)
Saved: vehicle_registry_2024_2026 (861136 rows)
Saved: wltp_models_2024_2026 (19847 rows)


## 30. Postrges DB connection

In [ ]:
from sqlalchemy import create_engine, text, Date
from getpass import getpass

password = getpass("PostgreSQL password: ")
db_config = {
    "host": "127.0.0.1",
    "port": 5432,
    "user": "postgres",
    "password": password
}
# Connect to the default database
admin_engine = create_engine(
    "postgresql+psycopg2://",
    connect_args={**db_config, "dbname": "postgres"},
    isolation_level="AUTOCOMMIT"
)
# Create database if needed
with admin_engine.connect() as conn:
    exists = conn.execute(
        text("SELECT 1 FROM pg_database WHERE datname = :name"),
        {"name": "voltway_israel"}
    ).scalar()

    if not exists:
        conn.execute(text("CREATE DATABASE voltway_israel"))
        print("Database created: voltway_israel")
    else:
        print("Database already exists")

admin_engine.dispose()

# Connect to our project database

engine = create_engine(
    "postgresql+psycopg2://",
    connect_args={**db_config, "dbname": "voltway_israel"}
)

with engine.connect() as conn:
    result = conn.execute(text("SELECT current_database()"))
    print("Connected to:", result.scalar())

Database already exists
Connected to: voltway_israel


## 31. Database fill

In [30]:
vehicles_db = vehicles.copy()

date_columns = [
    "first_registration_date",
    "mivchan_acharon_dt",
    "tokef_dt"
]

for col in date_columns:
    vehicles_db[col] = pd.to_datetime(
        vehicles_db[col],
        errors="coerce"
    )

vehicles_db["registration_quarter"] = (
    vehicles_db["registration_quarter"].astype("string")
)

print(vehicles_db[date_columns].dtypes)
print(vehicles_db[date_columns].isna().sum())

vehicles_db.to_sql(
    "vehicle_registry",
    engine,
    schema="public",
    if_exists="replace",
    index=False,
    chunksize=5000,
    method="multi",
    dtype={col: Date() for col in date_columns}
)

first_registration_date    datetime64[us]
mivchan_acharon_dt         datetime64[us]
tokef_dt                   datetime64[us]
dtype: object
first_registration_date    433
mivchan_acharon_dt           0
tokef_dt                     0
dtype: int64


861136

## 32. Vehicles check before DB export

In [31]:
for col in date_columns:
    print(
        col,
        "Before:", vehicles[col].isna().sum(),
        "After:", vehicles_db[col].isna().sum()
    )

first_registration_date Before: 433 After: 433
mivchan_acharon_dt Before: 0 After: 0
tokef_dt Before: 0 After: 0


## 33. DB big export check

In [32]:
with engine.connect() as conn:
    count = conn.execute(
        text("SELECT COUNT(*) FROM public.vehicle_registry")
    ).scalar()

print("Rows in PostgreSQL:", count)

assert count == len(vehicles_db)

Rows in PostgreSQL: 861136


## 34. Export remaining tables

In [33]:
for name, df in export_tables.items():
    if name == "vehicle_registry_2024_2026":
        continue

    df.to_sql(
        name,
        engine,
        schema="public",
        if_exists="replace",
        index=False,
        chunksize=500
    )

    print(f"Uploaded: {name} ({len(df)} rows)")


Uploaded: columns_vehicle_registry_en (22 rows)
Uploaded: ev_by_district (90 rows)
Uploaded: lookup_districts (27 rows)
Uploaded: lookup_fuel (6 rows)
Uploaded: lookup_makers (225 rows)
Uploaded: lookup_ownership (5 rows)
Uploaded: lookup_wltp_body_type (14 rows)
Uploaded: lookup_wltp_technology (4 rows)
Uploaded: wltp_models_2024_2026 (19847 rows)


## 35. Final validation

In [34]:
from sqlalchemy import inspect, text

inspector = inspect(engine)

with engine.connect() as conn:
    for name, df in export_tables.items():
        table = (
            "vehicle_registry"
            if name == "vehicle_registry_2024_2026"
            else name
        )

        assert inspector.has_table(table, schema="public")

        count = conn.execute(
            text(f'SELECT COUNT(*) FROM public."{table}"')
        ).scalar()

        assert count == len(df), f"{table}: row count mismatch"

        print(f"OK: {table} — {count} rows")

print("\nAll 10 tables validated!")

OK: columns_vehicle_registry_en — 22 rows
OK: ev_by_district — 90 rows
OK: lookup_districts — 27 rows
OK: lookup_fuel — 6 rows
OK: lookup_makers — 225 rows
OK: lookup_ownership — 5 rows
OK: lookup_wltp_body_type — 14 rows
OK: lookup_wltp_technology — 4 rows
OK: vehicle_registry — 861136 rows
OK: wltp_models_2024_2026 — 19847 rows

All 10 tables validated!


## 36. Final data notes
We finished preparing the data for the VoltWay Israel project.

All 10 tables were loaded into PostgreSQL database voltway_israel.

We checked the number of rows in all tables. Everything matches our prepared data.

Important notes:

The vehicle registry has 861,136 cars. Each car has a unique registration number. No cars were removed.

There are 163,393 electric cars in the registry. Hybrid cars are counted separately.

We have 433 cars without a first registration date. We cannot use them for quarterly analysis.

Some technical fields still have missing values. We decided to keep them because they are not important for our main questions.

We restored missing production countries using WLTP data.

The district table has 90 rows. Some locations are missing. We kept these records and did not guess their locations.

The WLTP table has 19,847 rows. Some maker and model codes are repeated, but there are no completely duplicated rows.

We should not join WLTP and vehicle registry using only maker and model codes. This could create extra rows and incorrect results.

We added English names for fuel types, manufacturers, ownership types, vehicle colors, districts, body types and drive technologies.

All prepared CSV files are saved in the clean_data folder. Original files are kept in the data folder.

Final result:

The data is ready for the next stage of analysis in PostgreSQL. We kept the original records and documented the known data limitations.